In [ ]:
import urllib.request, zipfile, io, os

DATA_DIR = r"C:\Users\brand\OneDrive\Imágenes\Escritorio\Git Portfolio\bank-marketing-mlop\data_raw"
os.makedirs(DATA_DIR, exist_ok=True)

url = 'https://archive.ics.uci.edu/static/public/222/bank+marketing.zip'
zip_path = os.path.join(DATA_DIR, 'bank_marketing.zip')

urllib.request.urlretrieve(url, zip_path)

with zipfile.ZipFile(zip_path, 'r') as outer:
    inner_bytes = outer.read('bank-additional.zip')

with zipfile.ZipFile(io.BytesIO(inner_bytes), 'r') as inner:
    csv_bytes = inner.read('bank-additional/bank-additional-full.csv')

with open(os.path.join(DATA_DIR, 'bank-additional-full.csv'), 'wb') as f:
    f.write(csv_bytes)

os.remove(zip_path)  # clean up the zip after extraction
print(f"Dataset ready at: {DATA_DIR}")

Dataset ready at: C:\Users\brand\OneDrive\Imágenes\Escritorio\Git Portfolio\bank-marketing-mlop\data_raw


In [48]:
import pandas as pd
import numpy as np


df = pd.read_csv(r'C:\Users\brand\OneDrive\Imágenes\Escritorio\Git Portfolio\bank-marketing-mlop\data_raw\bank-additional-full.csv', sep=';')
print(df.shape)           # Expected: (41188, 21)
print(df.dtypes)
print(df['y'].value_counts(normalize=True))  # Class distribution
df.head()


(41188, 21)
age                 int64
job                   str
marital               str
education             str
default               str
housing               str
loan                  str
contact               str
month                 str
day_of_week           str
duration            int64
campaign            int64
pdays               int64
previous            int64
poutcome              str
emp.var.rate      float64
cons.price.idx    float64
cons.conf.idx     float64
euribor3m         float64
nr.employed       float64
y                     str
dtype: object
y
no     0.887346
yes    0.112654
Name: proportion, dtype: float64


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


In [50]:
from sklearn.preprocessing import LabelEncoder


df_clean = df.copy()


# Encode binary target
df_clean['y'] = (df_clean['y'] == 'yes').astype(int)


# Drop 'duration' — it is a data leakage column (known only after call ends)
df_clean.drop(columns=['duration'], inplace=True)


# One-hot encode categoricals
cat_cols = df_clean.select_dtypes(include='object').columns.tolist()
df_encoded = pd.get_dummies(df_clean, columns=cat_cols, drop_first=True)


print(f'Features after encoding: {df_encoded.shape[1] - 1}')
df_encoded.head()


Features after encoding: 52


C:\Users\brand\AppData\Local\Temp\ipykernel_24400\3874578700.py:16: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df_clean.select_dtypes(include='object').columns.tolist()


,age,campaign,pdays,previous,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y,...,month_may,month_nov,month_oct,month_sep,day_of_week_mon,day_of_week_thu,day_of_week_tue,day_of_week_wed,poutcome_nonexistent,poutcome_success
0,56,1,999,0,1.1,93.994,-36.4,4.857,5191.0,0,...,True,False,False,False,True,False,False,False,True,False
1,57,1,999,0,1.1,93.994,-36.4,4.857,5191.0,0,...,True,False,False,False,True,False,False,False,True,False
2,37,1,999,0,1.1,93.994,-36.4,4.857,5191.0,0,...,True,False,False,False,True,False,False,False,True,False
3,40,1,999,0,1.1,93.994,-36.4,4.857,5191.0,0,...,True,False,False,False,True,False,False,False,True,False
4,56,1,999,0,1.1,93.994,-36.4,4.857,5191.0,0,...,True,False,False,False,True,False,False,False,True,False


In [51]:
from sklearn.model_selection import train_test_split


X = df_encoded.drop(columns=['y'])
y = df_encoded['y']


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


# Save for use in later notebooks
X_train.to_csv('X_train.csv', index=False)
X_test.to_csv('X_test.csv', index=False)
y_train.to_csv('y_train.csv', index=False)
y_test.to_csv('y_test.csv', index=False)


print(f'Train: {X_train.shape}, Test: {X_test.shape}')

Train: (32950, 52), Test: (8238, 52)
